
# Experiment 18 — Strong Official PatchTST Baselines on Weather and Electricity

## 목적

이번 실험에서는 retrieval을 붙이지 않습니다.

먼저 Weather와 Electricity에서 공식 PatchTST supervised recipe를 사용하여
강한 direct forecasting baseline을 확정합니다.

$$
H \in \{96, 192, 336, 720\}
$$

그리고 이후 Experiment 19에서 이 checkpoint를 **그대로 고정한 상태로**
기존 frozen historical-memory module을 적용합니다.

---

## 왜 이 실험을 분리하는가?

이전 screening에서는 모든 데이터셋에 거의 동일한 PatchTST 설정을 사용했습니다.
하지만 공식 PatchTST는 데이터셋마다 capacity와 optimization recipe가 다릅니다.

따라서 다음 비교를 하기 전에

$$
\boxed{
\text{Strong PatchTST} + \text{Historical Memory}
\quad \text{vs.} \quad
\text{Strong PatchTST}
}
$$

먼저 Strong PatchTST 자체를 독립적으로 확정해야 합니다.

---

## Official Weather recipe

공식 `weather.sh`를 따릅니다.

$$
L = 336
$$

- channels: 21
- encoder layers: 3
- heads: 16
- model dimension: 128
- FFN dimension: 256
- dropout: 0.2
- FC dropout: 0.2
- patch length: 16
- stride: 8
- batch size: 128
- learning rate: \(10^{-4}\)
- epochs: 100
- patience: 20
- seed: 2021
- LR schedule: default `type3`
- RevIN: on
- RevIN affine: off

---

## Official Electricity recipe

공식 `electricity.sh`를 따릅니다.

$$
L = 336
$$

- channels: 321
- encoder layers: 3
- heads: 16
- model dimension: 128
- FFN dimension: 256
- dropout: 0.2
- FC dropout: 0.2
- patch length: 16
- stride: 8
- batch size: 32
- learning rate: \(10^{-4}\)
- epochs: 100
- patience: 10
- seed: 2021
- LR schedule: `TST`
- OneCycleLR `pct_start=0.2`
- RevIN: on
- RevIN affine: off

---

## Data protocol

두 데이터셋 모두 공식 `Dataset_Custom` protocol을 재현합니다.

전체 길이를 \(N\)이라 할 때,

$$
N_{\text{train}} = \lfloor 0.7N \rfloor
$$

$$
N_{\text{test}} = \lfloor 0.2N \rfloor
$$

$$
N_{\text{val}}
=
N-N_{\text{train}}-N_{\text{test}}
$$

StandardScaler는 **train 구간에만 fit**합니다.

Validation/test에서는 input context가 split boundary 이전으로 이어질 수 있도록
공식 구현처럼 `seq_len`만큼 border를 뒤로 확장합니다.

---

## 두 종류의 test metric

### OfficialStyle

공식 `data_factory.py`와 동일하게

- `shuffle=False`
- `drop_last=True`

로 평가합니다.

### FullStride1

향후 retrieval augmentation과 동일한 sample set 비교를 위해

- 모든 valid test origin
- stride 1
- `drop_last=False`

로도 평가합니다.

Experiment 19의 direct baseline으로는 반드시 **FullStride1**을 사용합니다.


In [1]:

from pathlib import Path
from types import SimpleNamespace
import gc
import importlib
import json
import os
import random
import subprocess
import sys
import time
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 420)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

DATASETS = ["Weather", "Electricity"]
HORIZONS = [96, 192, 336, 720]

SEQ_LEN = 336
LABEL_LEN = 48
SEED = 2021

RESULT_ROOT = Path(
    "/data/dataset/strong_forecaster/"
    "weather_electricity_official_patchtst_baselines"
)

CHECKPOINT_DIR = RESULT_ROOT / "full_direct"
HISTORY_DIR = RESULT_ROOT / "history"
ARTIFACT_DIR = RESULT_ROOT / "artifacts"

for p in [
    RESULT_ROOT,
    CHECKPOINT_DIR,
    HISTORY_DIR,
    ARTIFACT_DIR,
]:
    p.mkdir(parents=True, exist_ok=True)

RESUME = True
FORCE_RETRAIN = False

print("Device:", DEVICE)
print("PyTorch:", torch.__version__)
print("Output:", RESULT_ROOT)


Device: cuda
PyTorch: 2.12.1+cu126
Output: /data/dataset/strong_forecaster/weather_electricity_official_patchtst_baselines


## 1. Locate the official PatchTST repository

In [2]:

OFFICIAL_REPO_CANDIDATES = [
    Path(
        "/code/stock_regime_retrieval/"
        "strong_forecaster/PatchTST_official"
    ),
    Path("/code/PatchTST_official"),
    Path("/data/PatchTST_official"),
    Path("/data/PatchTST"),
]

OFFICIAL_REPO = next(
    (
        p for p in OFFICIAL_REPO_CANDIDATES
        if (
            p
            / "PatchTST_supervised"
            / "models"
            / "PatchTST.py"
        ).exists()
    ),
    None,
)

if OFFICIAL_REPO is None:
    OFFICIAL_REPO = OFFICIAL_REPO_CANDIDATES[0]
    OFFICIAL_REPO.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    print(
        "Official repository not found. Cloning:",
        OFFICIAL_REPO,
    )

    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "https://github.com/yuqinie98/PatchTST.git",
            str(OFFICIAL_REPO),
        ],
        check=True,
    )

SUPERVISED_ROOT = (
    OFFICIAL_REPO
    / "PatchTST_supervised"
)

if not (
    SUPERVISED_ROOT
    / "models"
    / "PatchTST.py"
).exists():
    raise FileNotFoundError(
        f"Official supervised model not found: {SUPERVISED_ROOT}"
    )

try:
    commit = subprocess.check_output(
        [
            "git",
            "-C",
            str(OFFICIAL_REPO),
            "rev-parse",
            "HEAD",
        ],
        text=True,
    ).strip()
except Exception:
    commit = "unknown"

print("Official repo:", OFFICIAL_REPO)
print("Supervised root:", SUPERVISED_ROOT)
print("Git commit:", commit)

(ARTIFACT_DIR / "official_repo_commit.txt").write_text(
    commit + "\n"
)


Official repo: /code/stock_regime_retrieval/strong_forecaster/PatchTST_official
Supervised root: /code/stock_regime_retrieval/strong_forecaster/PatchTST_official/PatchTST_supervised
Git commit: 204c21efe0b39603ad6e2ca640ef5896646ab1a9


41

## 2. Import the official PatchTST model only

In [3]:

# Avoid accidental import of Time-Series-Library modules
# with the same top-level names.
for module_name in list(sys.modules.keys()):
    if (
        module_name == "models"
        or module_name.startswith("models.")
        or module_name == "layers"
        or module_name.startswith("layers.")
    ):
        del sys.modules[module_name]

if str(SUPERVISED_ROOT) in sys.path:
    sys.path.remove(str(SUPERVISED_ROOT))

sys.path.insert(0, str(SUPERVISED_ROOT))

patchtst_module = importlib.import_module(
    "models.PatchTST"
)

OfficialPatchTST = patchtst_module.Model

actual_model_file = Path(
    patchtst_module.__file__
).resolve()

expected_model_file = (
    SUPERVISED_ROOT
    / "models"
    / "PatchTST.py"
).resolve()

print("Imported model:", actual_model_file)

if actual_model_file != expected_model_file:
    raise RuntimeError(
        "Wrong PatchTST implementation imported.\n"
        f"Expected: {expected_model_file}\n"
        f"Actual:   {actual_model_file}"
    )

print(
    "PASS: official PatchTST supervised implementation is active."
)


Imported model: /code/stock_regime_retrieval/strong_forecaster/PatchTST_official/PatchTST_supervised/models/PatchTST.py
PASS: official PatchTST supervised implementation is active.



## 3. Dataset paths

아래 셀은 recursive filesystem search를 하지 않습니다.
기존 실험에서 사용한 일반적인 데이터 위치만 확인하므로 빠르게 실행됩니다.

데이터를 다른 위치에 두었다면 `DATA_PATH_CANDIDATES`에 해당 경로만 추가하면 됩니다.


In [4]:

DATA_PATH_CANDIDATES = {
    "Weather": [
        Path("/data/dataset/weather.csv"),
        Path("/data/dataset/weather/weather.csv"),
        Path("/data/Time-Series-Library/dataset/weather/weather.csv"),
        Path("/data/Time-Series-Library/dataset/weather.csv"),
        Path("/data/Time-Series-Library_v2/dataset/weather/weather.csv"),
        Path("/data/Time-Series-Library_v2/dataset/weather.csv"),
        Path("/code/stock_regime_retrieval/strong_forecaster/data/weather.csv"),
    ],
    "Electricity": [
        Path("/data/dataset/electricity.csv"),
        Path("/data/dataset/electricity/electricity.csv"),
        Path("/data/Time-Series-Library/dataset/electricity/electricity.csv"),
        Path("/data/Time-Series-Library/dataset/electricity.csv"),
        Path("/data/Time-Series-Library_v2/dataset/electricity/electricity.csv"),
        Path("/data/Time-Series-Library_v2/dataset/electricity.csv"),
        Path("/code/stock_regime_retrieval/strong_forecaster/data/electricity.csv"),
    ],
}

DATA_PATHS = {}

for name, candidates in DATA_PATH_CANDIDATES.items():
    hit = next(
        (p for p in candidates if p.is_file()),
        None,
    )

    DATA_PATHS[name] = hit

    print(
        f"{name:11s}:",
        hit if hit is not None else "NOT FOUND",
    )

missing = [
    name
    for name, path in DATA_PATHS.items()
    if path is None
]

if missing:
    print("\nAttempted paths:")
    for name in missing:
        print(f"\n{name}")
        for p in DATA_PATH_CANDIDATES[name]:
            print(" -", p)

    raise FileNotFoundError(
        "Dataset file(s) not found: "
        + ", ".join(missing)
        + ". Add the actual path to DATA_PATH_CANDIDATES."
    )


Weather    : /data/Time-Series-Library/dataset/weather/weather.csv
Electricity: /data/Time-Series-Library/dataset/electricity/electricity.csv


## 4. Reproduce the official `Dataset_Custom` data protocol

In [5]:

EXPECTED_CHANNELS = {
    "Weather": 21,
    "Electricity": 321,
}


def prepare_custom_data(name):
    path = DATA_PATHS[name]
    df_raw = pd.read_csv(path)

    if "date" not in df_raw.columns:
        raise ValueError(
            f"{name}: official Dataset_Custom requires a 'date' column."
        )

    # Official Dataset_Custom moves target 'OT' to the final column.
    # For M forecasting this preserves the exact official column ordering.
    if "OT" not in df_raw.columns:
        raise ValueError(
            f"{name}: official script uses the default target='OT', "
            "but the CSV has no OT column. "
            f"First columns: {df_raw.columns.tolist()[:10]}"
        )

    cols = list(df_raw.columns)
    cols.remove("OT")
    cols.remove("date")

    ordered = df_raw[
        ["date"] + cols + ["OT"]
    ].copy()

    value_cols = list(
        ordered.columns[1:]
    )

    raw = ordered[
        value_cols
    ].to_numpy(
        dtype=np.float32
    )

    n = len(raw)

    num_train = int(n * 0.7)
    num_test = int(n * 0.2)
    num_val = n - num_train - num_test

    train_end = num_train
    val_end = num_train + num_val
    test_end = n

    scaler = StandardScaler()
    scaler.fit(
        raw[:train_end]
    )

    z = scaler.transform(
        raw
    ).astype(
        np.float32
    )

    expected_c = EXPECTED_CHANNELS[name]

    if raw.shape[1] != expected_c:
        raise ValueError(
            f"{name}: expected {expected_c} channels from the official "
            f"script but found {raw.shape[1]}."
        )

    return {
        "name": name,
        "path": path,
        "df": ordered,
        "columns": value_cols,
        "raw": raw,
        "z": z,
        "mean": scaler.mean_.astype(np.float32),
        "scale": scaler.scale_.astype(np.float32),
        "n_channels": raw.shape[1],
        "n": n,
        "train_end": train_end,
        "val_end": val_end,
        "test_end": test_end,
        "num_train": num_train,
        "num_val": num_val,
        "num_test": num_test,
    }


DATA = {
    name: prepare_custom_data(name)
    for name in DATASETS
}

protocol_df = pd.DataFrame([
    {
        "Dataset": name,
        "Path": str(d["path"]),
        "Rows": d["n"],
        "Channels": d["n_channels"],
        "TrainRows": d["num_train"],
        "ValRows": d["num_val"],
        "TestRows": d["num_test"],
        "TrainEnd": d["train_end"],
        "ValEnd": d["val_end"],
        "TestEnd": d["test_end"],
    }
    for name, d in DATA.items()
])

display(protocol_df)

protocol_df.to_csv(
    ARTIFACT_DIR / "dataset_protocol.csv",
    index=False,
)


,Dataset,Path,Rows,Channels,TrainRows,ValRows,TestRows,TrainEnd,ValEnd,TestEnd
0,Weather,/data/Time-Series-Library/dataset/weather/weat...,52696,21,36887,5270,10539,36887,42157,52696
1,Electricity,/data/Time-Series-Library/dataset/electricity/...,26304,321,18412,2632,5260,18412,21044,26304


## 5. Official recipes

In [6]:

RECIPES = {
    "Weather": {
        "enc_in": 21,
        "e_layers": 3,
        "n_heads": 16,
        "d_model": 128,
        "d_ff": 256,
        "dropout": 0.2,
        "fc_dropout": 0.2,
        "head_dropout": 0.0,
        "patch_len": 16,
        "stride": 8,
        "batch_size": 128,
        "train_epochs": 100,
        "patience": 20,
        "learning_rate": 1e-4,
        # weather.sh does not override these parser defaults.
        "lradj": "type3",
        "pct_start": 0.3,
        "seed": 2021,
    },
    "Electricity": {
        "enc_in": 321,
        "e_layers": 3,
        "n_heads": 16,
        "d_model": 128,
        "d_ff": 256,
        "dropout": 0.2,
        "fc_dropout": 0.2,
        "head_dropout": 0.0,
        "patch_len": 16,
        "stride": 8,
        "batch_size": 32,
        "train_epochs": 100,
        "patience": 10,
        "learning_rate": 1e-4,
        "lradj": "TST",
        "pct_start": 0.2,
        "seed": 2021,
    },
}

display(
    pd.DataFrame(RECIPES).T
)


,enc_in,e_layers,n_heads,d_model,d_ff,dropout,fc_dropout,head_dropout,patch_len,stride,batch_size,train_epochs,patience,learning_rate,lradj,pct_start,seed
Weather,21,3,16,128,256,0.2,0.2,0.0,16,8,128,100,20,0.0001,type3,0.3,2021
Electricity,321,3,16,128,256,0.2,0.2,0.0,16,8,32,100,10,0.0001,TST,0.2,2021



## 6. Local Dataset class with exact official borders

Validation과 test는 split boundary에서 바로 시작하는 forecast origin을 포함하기 위해
context만 이전 split에 걸쳐 사용합니다.

예를 들어 validation의 `border1`은

$$
N_{\text{train}} - L
$$

입니다.

하지만 target은 validation boundary 이후에만 위치하므로 label leakage는 없습니다.


In [7]:

class CustomForecastDataset(Dataset):
    def __init__(
        self,
        name,
        flag,
        horizon,
    ):
        super().__init__()

        if flag not in {
            "train",
            "val",
            "test",
        }:
            raise ValueError(flag)

        self.name = name
        self.flag = flag
        self.horizon = int(horizon)

        d = DATA[name]
        z = d["z"]

        border1s = [
            0,
            d["train_end"] - SEQ_LEN,
            d["val_end"] - SEQ_LEN,
        ]

        border2s = [
            d["train_end"],
            d["val_end"],
            d["test_end"],
        ]

        idx = {
            "train": 0,
            "val": 1,
            "test": 2,
        }[flag]

        self.border1 = int(border1s[idx])
        self.border2 = int(border2s[idx])

        self.data_x = z[
            self.border1:
            self.border2
        ]

    def __getitem__(self, index):
        s_begin = index
        s_end = s_begin + SEQ_LEN

        r_begin = s_end - LABEL_LEN
        r_end = (
            r_begin
            + LABEL_LEN
            + self.horizon
        )

        seq_x = self.data_x[
            s_begin:s_end
        ]

        seq_y = self.data_x[
            r_begin:r_end
        ]

        return (
            torch.from_numpy(seq_x),
            torch.from_numpy(seq_y),
        )

    def __len__(self):
        return (
            len(self.data_x)
            - SEQ_LEN
            - self.horizon
            + 1
        )


window_rows = []

for name in DATASETS:
    for h in HORIZONS:
        for flag in [
            "train",
            "val",
            "test",
        ]:
            ds = CustomForecastDataset(
                name,
                flag,
                h,
            )

            window_rows.append({
                "Dataset": name,
                "Horizon": h,
                "Split": flag,
                "Windows": len(ds),
                "Border1": ds.border1,
                "Border2": ds.border2,
            })

display(
    pd.DataFrame(window_rows)
)


,Dataset,Horizon,Split,Windows,Border1,Border2
0,Weather,96,train,36456,0,36887
1,Weather,96,val,5175,36551,42157
2,Weather,96,test,10444,41821,52696
3,Weather,192,train,36360,0,36887
4,Weather,192,val,5079,36551,42157
5,Weather,192,test,10348,41821,52696
6,Weather,336,train,36216,0,36887
7,Weather,336,val,4935,36551,42157
8,Weather,336,test,10204,41821,52696
9,Weather,720,train,35832,0,36887


## 7. Build official PatchTST

In [8]:

def make_config(name, horizon):
    r = RECIPES[name]

    return SimpleNamespace(
        enc_in=r["enc_in"],
        seq_len=SEQ_LEN,
        pred_len=int(horizon),
        e_layers=r["e_layers"],
        n_heads=r["n_heads"],
        d_model=r["d_model"],
        d_ff=r["d_ff"],
        dropout=r["dropout"],
        fc_dropout=r["fc_dropout"],
        head_dropout=r["head_dropout"],
        individual=0,
        patch_len=r["patch_len"],
        stride=r["stride"],
        padding_patch="end",
        revin=1,
        affine=0,
        subtract_last=0,
        decomposition=0,
        kernel_size=25,
    )


def build_model(name, horizon):
    model = OfficialPatchTST(
        make_config(
            name,
            horizon,
        )
    ).float().to(DEVICE)

    return model


param_rows = []

for name in DATASETS:
    for h in HORIZONS:
        model = build_model(name, h)

        params = sum(
            p.numel()
            for p in model.parameters()
            if p.requires_grad
        )

        param_rows.append({
            "Dataset": name,
            "Horizon": h,
            "TrainableParams": params,
            "TrainableParams_M": params / 1e6,
        })

        del model

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

display(
    pd.DataFrame(param_rows)
)


,Dataset,Horizon,TrainableParams,TrainableParams_M
0,Weather,96,921184,0.921184
1,Weather,192,1437376,1.437376
2,Weather,336,2211664,2.211664
3,Weather,720,4276432,4.276432
4,Electricity,96,921184,0.921184
5,Electricity,192,1437376,1.437376
6,Electricity,336,2211664,2.211664
7,Electricity,720,4276432,4.276432


## 8. DataLoaders

In [9]:

def make_loaders(name, horizon):
    r = RECIPES[name]

    train_ds = CustomForecastDataset(
        name,
        "train",
        horizon,
    )

    val_ds = CustomForecastDataset(
        name,
        "val",
        horizon,
    )

    test_ds = CustomForecastDataset(
        name,
        "test",
        horizon,
    )

    # Official data_factory:
    # train/val: shuffle=True, drop_last=True
    # test: shuffle=False, drop_last=True
    train_loader = DataLoader(
        train_ds,
        batch_size=r["batch_size"],
        shuffle=True,
        num_workers=0,
        drop_last=True,
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=r["batch_size"],
        shuffle=True,
        num_workers=0,
        drop_last=True,
    )

    official_test_loader = DataLoader(
        test_ds,
        batch_size=r["batch_size"],
        shuffle=False,
        num_workers=0,
        drop_last=True,
    )

    # Our matched evaluation for Experiment 19.
    full_test_loader = DataLoader(
        test_ds,
        batch_size=r["batch_size"],
        shuffle=False,
        num_workers=0,
        drop_last=False,
    )

    return {
        "train_ds": train_ds,
        "val_ds": val_ds,
        "test_ds": test_ds,
        "train": train_loader,
        "val": val_loader,
        "official_test": official_test_loader,
        "full_test": full_test_loader,
    }


## 9. Evaluation

In [10]:

@torch.no_grad()
def evaluate_loader(
    model,
    loader,
    horizon,
):
    model.eval()

    batch_mse = []

    sse = 0.0
    sae = 0.0
    count = 0
    windows = 0

    for batch_x, batch_y in loader:
        x = batch_x.float().to(DEVICE)

        y = batch_y[
            :,
            -horizon:,
            :
        ].float().to(DEVICE)

        pred = model(x)[
            :,
            -horizon:,
            :
        ]

        err = pred - y

        batch_mse.append(
            float(
                (err ** 2).mean().item()
            )
        )

        sse += float(
            (err ** 2).sum().item()
        )

        sae += float(
            err.abs().sum().item()
        )

        count += err.numel()
        windows += len(x)

        del x, y, pred, err

    return {
        "BatchAverageMSE": float(
            np.mean(batch_mse)
        ),
        "MSE": sse / count,
        "MAE": sae / count,
        "Elements": count,
        "Batches": len(batch_mse),
        "Windows": windows,
    }


## 10. Official learning-rate behavior

In [11]:

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # Keep behavior close to the official script.
    torch.backends.cudnn.benchmark = False


def adjust_type3_lr(
    optimizer,
    base_lr,
    epoch,
):
    # Official PatchTST utils/tools.py type3 rule.
    lr = (
        base_lr
        if epoch < 3
        else base_lr
        * (
            0.9
            ** (
                epoch - 3
            )
        )
    )

    for group in optimizer.param_groups:
        group["lr"] = lr

    return lr


## 11. Checkpoint paths

In [12]:

def checkpoint_path(
    name,
    horizon,
):
    safe = name.lower()

    return (
        CHECKPOINT_DIR
        / (
            f"{safe}_L336_H{horizon}_"
            "official_recipe_seed2021.pt"
        )
    )


def history_path(
    name,
    horizon,
):
    safe = name.lower()

    return (
        HISTORY_DIR
        / (
            f"{safe}_L336_H{horizon}_"
            "official_recipe_seed2021.csv"
        )
    )



## 12. Training loop

Model selection에는 validation loss만 사용합니다.

공식 training code는 epoch마다 test loss도 출력하지만,
이번 notebook에서는 test를 training 중에 평가하지 않습니다.

이는 optimization이나 checkpoint selection을 바꾸지 않으면서
test inspection을 피하기 위한 더 엄격한 protocol입니다.


In [13]:

def train_or_load(
    name,
    horizon,
):
    r = RECIPES[name]
    ckpt_path = checkpoint_path(
        name,
        horizon,
    )

    if (
        RESUME
        and ckpt_path.exists()
        and not FORCE_RETRAIN
    ):
        ckpt = torch.load(
            ckpt_path,
            map_location=DEVICE,
        )

        model = build_model(
            name,
            horizon,
        )

        model.load_state_dict(
            ckpt["StateDict"]
        )

        model.eval()

        print(
            f"Loaded {name} H={horizon}: "
            f"best={ckpt['BestValMSE']:.6f}"
            f"@{ckpt['BestEpoch']}"
        )

        return (
            model,
            ckpt,
            make_loaders(
                name,
                horizon,
            ),
        )

    set_seed(
        r["seed"]
    )

    loaders = make_loaders(
        name,
        horizon,
    )

    model = build_model(
        name,
        horizon,
    )

    # Official exp_main.py uses Adam.
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=r["learning_rate"],
    )

    onecycle = torch.optim.lr_scheduler.OneCycleLR(
        optimizer=optimizer,
        steps_per_epoch=len(
            loaders["train"]
        ),
        pct_start=r["pct_start"],
        epochs=r["train_epochs"],
        max_lr=r["learning_rate"],
    )

    best_val = float("inf")
    best_epoch = -1
    best_state = None
    wait = 0
    history = []

    for epoch in range(
        1,
        r["train_epochs"] + 1,
    ):
        model.train()

        losses = []
        t0 = time.time()

        for batch_x, batch_y in loaders["train"]:
            x = batch_x.float().to(DEVICE)

            y = batch_y[
                :,
                -horizon:,
                :
            ].float().to(DEVICE)

            optimizer.zero_grad(
                set_to_none=True
            )

            pred = model(x)[
                :,
                -horizon:,
                :
            ]

            loss = F.mse_loss(
                pred,
                y,
            )

            loss.backward()
            optimizer.step()

            losses.append(
                float(
                    loss.item()
                )
            )

            if r["lradj"] == "TST":
                # Match official PatchTST ordering:
                # read current OneCycle LR, assign it,
                # then advance scheduler.
                current = onecycle.get_last_lr()[0]

                for group in optimizer.param_groups:
                    group["lr"] = current

                onecycle.step()

            del x, y, pred, loss

        train_mse = float(
            np.mean(losses)
        )

        val = evaluate_loader(
            model,
            loaders["val"],
            horizon,
        )

        # Official EarlyStopping receives mean validation batch MSE.
        val_for_selection = val[
            "BatchAverageMSE"
        ]

        if (
            val_for_selection
            < best_val
            - 1e-12
        ):
            best_val = val_for_selection
            best_epoch = epoch

            best_state = {
                k:
                    v.detach()
                    .cpu()
                    .clone()
                for k, v
                in model.state_dict().items()
            }

            wait = 0
        else:
            wait += 1

        if r["lradj"] != "TST":
            lr = adjust_type3_lr(
                optimizer,
                r["learning_rate"],
                epoch,
            )
        else:
            lr = onecycle.get_last_lr()[0]

        history.append({
            "Epoch": epoch,
            "TrainMSE": train_mse,
            "ValBatchAverageMSE": val_for_selection,
            "ValGlobalMSE": val["MSE"],
            "ValMAE": val["MAE"],
            "BestValMSE": best_val,
            "BestEpoch": best_epoch,
            "LR": lr,
            "Seconds": time.time() - t0,
        })

        print(
            f"{name:11s} H={horizon:3d} "
            f"ep={epoch:03d}/{r['train_epochs']} | "
            f"train={train_mse:.6f} | "
            f"val={val_for_selection:.6f} | "
            f"best={best_val:.6f}@{best_epoch} | "
            f"lr={lr:.3e} | "
            f"wait={wait}/{r['patience']}"
        )

        pd.DataFrame(
            history
        ).to_csv(
            history_path(
                name,
                horizon,
            ),
            index=False,
        )

        if wait >= r["patience"]:
            print(
                f"Early stopping: {name} H={horizon} "
                f"at epoch {epoch}."
            )
            break

    if best_state is None:
        raise RuntimeError(
            f"{name} H={horizon}: no checkpoint selected."
        )

    model.load_state_dict(
        best_state
    )

    model.eval()

    ckpt = {
        "Dataset": name,
        "SeqLen": SEQ_LEN,
        "PredLen": int(horizon),
        "Seed": r["seed"],
        "Recipe": r,
        "OfficialRepoCommit": commit,
        "BestEpoch": int(best_epoch),
        "BestValMSE": float(best_val),
        "StateDict": best_state,
    }

    torch.save(
        ckpt,
        ckpt_path,
    )

    return (
        model,
        ckpt,
        loaders,
    )



## 13. Run all eight strong baselines

실행 순서는 다음과 같습니다.

$$
\text{Weather: }96 \rightarrow 192 \rightarrow 336 \rightarrow 720
$$

$$
\text{Electricity: }96 \rightarrow 192 \rightarrow 336 \rightarrow 720
$$

각 조건이 완료될 때마다 checkpoint와 summary를 저장합니다.
따라서 중간에 커널이 종료되어도 `RESUME=True`로 이어갈 수 있습니다.


In [14]:

SUMMARY_PATH = (
    RESULT_ROOT
    / "summary.csv"
)

existing = (
    pd.read_csv(
        SUMMARY_PATH
    )
    if (
        RESUME
        and SUMMARY_PATH.exists()
    )
    else pd.DataFrame()
)

result_rows = (
    existing.to_dict("records")
    if len(existing)
    else []
)


def already_done(
    name,
    horizon,
):
    if not len(existing):
        return False

    return bool(
        (
            (existing["Dataset"] == name)
            & (
                existing["Horizon"]
                == horizon
            )
        ).any()
    )


for name in DATASETS:
    for horizon in HORIZONS:
        if already_done(
            name,
            horizon,
        ):
            print(
                f"SKIP completed: {name} H={horizon}"
            )
            continue

        print(
            "\n"
            + "=" * 140
        )

        print(
            f"OFFICIAL PATCHTST | "
            f"{name} | L=336 | H={horizon}"
        )

        print(
            "=" * 140
        )

        t0 = time.time()

        model, ckpt, loaders = train_or_load(
            name,
            horizon,
        )

        official_style = evaluate_loader(
            model,
            loaders["official_test"],
            horizon,
        )

        full_stride1 = evaluate_loader(
            model,
            loaders["full_test"],
            horizon,
        )

        row = {
            "Dataset": name,
            "Horizon": horizon,
            "SeqLen": SEQ_LEN,
            "Channels": DATA[name]["n_channels"],
            "BestEpoch": ckpt["BestEpoch"],
            "BestValMSE": ckpt["BestValMSE"],
            "OfficialStyle_MSE": official_style["MSE"],
            "OfficialStyle_MAE": official_style["MAE"],
            "OfficialStyle_BatchAverageMSE": (
                official_style["BatchAverageMSE"]
            ),
            "OfficialStyle_Windows": (
                official_style["Windows"]
            ),
            "FullStride1_MSE": full_stride1["MSE"],
            "FullStride1_MAE": full_stride1["MAE"],
            "FullStride1_Windows": (
                full_stride1["Windows"]
            ),
            "ExcludedByOfficialDropLast": (
                full_stride1["Windows"]
                - official_style["Windows"]
            ),
            "RuntimeMinutes": (
                time.time() - t0
            ) / 60.0,
            "Checkpoint": str(
                checkpoint_path(
                    name,
                    horizon,
                )
            ),
            "OfficialRepoCommit": commit,
        }

        result_rows.append(row)

        pd.DataFrame(
            result_rows
        ).to_csv(
            SUMMARY_PATH,
            index=False,
        )

        display(
            pd.DataFrame([row])[
                [
                    "Dataset",
                    "Horizon",
                    "BestEpoch",
                    "BestValMSE",
                    "OfficialStyle_MSE",
                    "OfficialStyle_MAE",
                    "FullStride1_MSE",
                    "FullStride1_MAE",
                    "FullStride1_Windows",
                    "ExcludedByOfficialDropLast",
                ]
            ]
        )

        del (
            model,
            ckpt,
            loaders,
        )

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()


summary_df = (
    pd.DataFrame(result_rows)
    .sort_values(
        [
            "Dataset",
            "Horizon",
        ]
    )
    .reset_index(drop=True)
)

display(summary_df)



OFFICIAL PATCHTST | Weather | L=336 | H=96
Weather     H= 96 ep=001/100 | train=0.747358 | val=0.542040 | best=0.542040@1 | lr=1.000e-04 | wait=0/20
Weather     H= 96 ep=002/100 | train=0.492498 | val=0.418392 | best=0.418392@2 | lr=1.000e-04 | wait=0/20
Weather     H= 96 ep=003/100 | train=0.448917 | val=0.403692 | best=0.403692@3 | lr=1.000e-04 | wait=0/20
Weather     H= 96 ep=004/100 | train=0.441734 | val=0.395931 | best=0.395931@4 | lr=9.000e-05 | wait=0/20
Weather     H= 96 ep=005/100 | train=0.436128 | val=0.396592 | best=0.395931@4 | lr=8.100e-05 | wait=1/20
Weather     H= 96 ep=006/100 | train=0.432803 | val=0.395598 | best=0.395598@6 | lr=7.290e-05 | wait=0/20
Weather     H= 96 ep=007/100 | train=0.430140 | val=0.393127 | best=0.393127@7 | lr=6.561e-05 | wait=0/20
Weather     H= 96 ep=008/100 | train=0.427260 | val=0.397539 | best=0.393127@7 | lr=5.905e-05 | wait=1/20
Weather     H= 96 ep=009/100 | train=0.425754 | val=0.396876 | best=0.393127@7 | lr=5.314e-05 | wait=2/20
We

,Dataset,Horizon,BestEpoch,BestValMSE,OfficialStyle_MSE,OfficialStyle_MAE,FullStride1_MSE,FullStride1_MAE,FullStride1_Windows,ExcludedByOfficialDropLast
0,Weather,96,65,0.390229,0.150419,0.198269,0.14969,0.197816,10444,76



OFFICIAL PATCHTST | Weather | L=336 | H=192
Weather     H=192 ep=001/100 | train=0.779212 | val=0.597681 | best=0.597681@1 | lr=1.000e-04 | wait=0/20
Weather     H=192 ep=002/100 | train=0.537527 | val=0.489016 | best=0.489016@2 | lr=1.000e-04 | wait=0/20
Weather     H=192 ep=003/100 | train=0.501410 | val=0.471943 | best=0.471943@3 | lr=1.000e-04 | wait=0/20
Weather     H=192 ep=004/100 | train=0.494108 | val=0.470940 | best=0.470940@4 | lr=9.000e-05 | wait=0/20
Weather     H=192 ep=005/100 | train=0.489733 | val=0.463755 | best=0.463755@5 | lr=8.100e-05 | wait=0/20
Weather     H=192 ep=006/100 | train=0.487068 | val=0.461682 | best=0.461682@6 | lr=7.290e-05 | wait=0/20
Weather     H=192 ep=007/100 | train=0.484677 | val=0.462775 | best=0.461682@6 | lr=6.561e-05 | wait=1/20
Weather     H=192 ep=008/100 | train=0.482582 | val=0.463055 | best=0.461682@6 | lr=5.905e-05 | wait=2/20
Weather     H=192 ep=009/100 | train=0.480784 | val=0.466814 | best=0.461682@6 | lr=5.314e-05 | wait=3/20
W

,Dataset,Horizon,BestEpoch,BestValMSE,OfficialStyle_MSE,OfficialStyle_MAE,FullStride1_MSE,FullStride1_MAE,FullStride1_Windows,ExcludedByOfficialDropLast
0,Weather,192,45,0.456669,0.195386,0.241418,0.194775,0.240853,10348,108



OFFICIAL PATCHTST | Weather | L=336 | H=336
Weather     H=336 ep=001/100 | train=0.823969 | val=0.673787 | best=0.673787@1 | lr=1.000e-04 | wait=0/20
Weather     H=336 ep=002/100 | train=0.589323 | val=0.570568 | best=0.570568@2 | lr=1.000e-04 | wait=0/20
Weather     H=336 ep=003/100 | train=0.557555 | val=0.561700 | best=0.561700@3 | lr=1.000e-04 | wait=0/20
Weather     H=336 ep=004/100 | train=0.550439 | val=0.552647 | best=0.552647@4 | lr=9.000e-05 | wait=0/20
Weather     H=336 ep=005/100 | train=0.546296 | val=0.551869 | best=0.551869@5 | lr=8.100e-05 | wait=0/20
Weather     H=336 ep=006/100 | train=0.543647 | val=0.551974 | best=0.551869@5 | lr=7.290e-05 | wait=1/20
Weather     H=336 ep=007/100 | train=0.541770 | val=0.552601 | best=0.551869@5 | lr=6.561e-05 | wait=2/20
Weather     H=336 ep=008/100 | train=0.539138 | val=0.550212 | best=0.550212@8 | lr=5.905e-05 | wait=0/20
Weather     H=336 ep=009/100 | train=0.537121 | val=0.547726 | best=0.547726@9 | lr=5.314e-05 | wait=0/20
W

...


OFFICIAL PATCHTST | Electricity | L=336 | H=336
Electricity H=336 ep=001/100 | train=0.425764 | val=0.208078 | best=0.208078@1 | lr=4.591e-06 | wait=0/10
Electricity H=336 ep=002/100 | train=0.270780 | val=0.176862 | best=0.176862@2 | lr=6.350e-06 | wait=0/10
Electricity H=336 ep=003/100 | train=0.233985 | val=0.163517 | best=0.163517@3 | lr=9.233e-06 | wait=0/10
Electricity H=336 ep=004/100 | train=0.216319 | val=0.157306 | best=0.157306@4 | lr=1.317e-05 | wait=0/10
Electricity H=336 ep=005/100 | train=0.207416 | val=0.154013 | best=0.154013@5 | lr=1.806e-05 | wait=0/10
Electricity H=336 ep=006/100 | train=0.201879 | val=0.152332 | best=0.152332@6 | lr=2.379e-05 | wait=0/10
Electricity H=336 ep=007/100 | train=0.198015 | val=0.151333 | best=0.151333@7 | lr=3.021e-05 | wait=0/10
Electricity H=336 ep=008/100 | train=0.195158 | val=0.150490 | best=0.150490@8 | lr=3.717e-05 | wait=0/10
Electricity H=336 ep=009/100 | train=0.192871 | val=0.151096 | best=0.150490@8 | lr=4.450e-05 | wait=1/

,Dataset,Horizon,BestEpoch,BestValMSE,OfficialStyle_MSE,OfficialStyle_MAE,FullStride1_MSE,FullStride1_MAE,FullStride1_Windows,ExcludedByOfficialDropLast
0,Electricity,336,69,0.140645,0.164611,0.258211,0.16569,0.258694,4925,29



OFFICIAL PATCHTST | Electricity | L=336 | H=720
Electricity H=720 ep=001/100 | train=0.470779 | val=0.240102 | best=0.240102@1 | lr=4.591e-06 | wait=0/10
Electricity H=720 ep=002/100 | train=0.314628 | val=0.208828 | best=0.208828@2 | lr=6.350e-06 | wait=0/10
Electricity H=720 ep=003/100 | train=0.278951 | val=0.194973 | best=0.194973@3 | lr=9.233e-06 | wait=0/10
Electricity H=720 ep=004/100 | train=0.261796 | val=0.188743 | best=0.188743@4 | lr=1.317e-05 | wait=0/10
Electricity H=720 ep=005/100 | train=0.252910 | val=0.185612 | best=0.185612@5 | lr=1.806e-05 | wait=0/10
Electricity H=720 ep=006/100 | train=0.247512 | val=0.183264 | best=0.183264@6 | lr=2.379e-05 | wait=0/10
Electricity H=720 ep=007/100 | train=0.243755 | val=0.181605 | best=0.181605@7 | lr=3.021e-05 | wait=0/10
Electricity H=720 ep=008/100 | train=0.241070 | val=0.181211 | best=0.181211@8 | lr=3.717e-05 | wait=0/10
Electricity H=720 ep=009/100 | train=0.238516 | val=0.180988 | best=0.180988@9 | lr=4.450e-05 | wait=0/

,Dataset,Horizon,BestEpoch,BestValMSE,OfficialStyle_MSE,OfficialStyle_MAE,FullStride1_MSE,FullStride1_MAE,FullStride1_Windows,ExcludedByOfficialDropLast
0,Electricity,720,39,0.173104,0.202358,0.292115,0.203216,0.292492,4541,29


,Dataset,Horizon,SeqLen,Channels,BestEpoch,BestValMSE,OfficialStyle_MSE,OfficialStyle_MAE,OfficialStyle_BatchAverageMSE,OfficialStyle_Windows,FullStride1_MSE,FullStride1_MAE,FullStride1_Windows,ExcludedByOfficialDropLast,RuntimeMinutes,Checkpoint,OfficialRepoCommit
0,Electricity,96,336,321,97,0.110311,0.129935,0.222482,0.129935,5152,0.129977,0.222527,5165,13,340.099770,/data/dataset/strong_forecaster/weather_electr...,204c21efe0b39603ad6e2ca640ef5896646ab1a9
1,Electricity,192,336,321,48,0.126097,0.148336,0.241681,0.148336,5056,0.149116,0.242038,5069,13,196.938228,/data/dataset/strong_forecaster/weather_electr...,204c21efe0b39603ad6e2ca640ef5896646ab1a9
2,Electricity,336,336,321,69,0.140645,0.164611,0.258211,0.164611,4896,0.165690,0.258694,4925,29,269.017774,/data/dataset/strong_forecaster/weather_electr...,204c21efe0b39603ad6e2ca640ef5896646ab1a9
3,Electricity,720,336,321,39,0.173104,0.202358,0.292115,0.202358,4512,0.203216,0.292492,4541,29,168.304391,/data/dataset/strong_forecaster/weather_electr...,204c21efe0b39603ad6e2ca640ef5896646ab1a9
4,Weather,96,336,21,65,0.390229,0.150419,0.198269,0.150419,10368,0.149690,0.197816,10444,76,39.099661,/data/dataset/strong_forecaster/weather_electr...,204c21efe0b39603ad6e2ca640ef5896646ab1a9
5,Weather,192,336,21,45,0.456669,0.195386,0.241418,0.195386,10240,0.194775,0.240853,10348,108,30.519710,/data/dataset/strong_forecaster/weather_electr...,204c21efe0b39603ad6e2ca640ef5896646ab1a9
6,Weather,336,336,21,19,0.547002,0.247370,0.282072,0.247370,10112,0.246900,0.281548,10204,92,18.725101,/data/dataset/strong_forecaster/weather_electr...,204c21efe0b39603ad6e2ca640ef5896646ab1a9
7,Weather,720,336,21,23,0.653166,0.319053,0.334466,0.319053,9728,0.320894,0.334351,9820,92,21.177871,/data/dataset/strong_forecaster/weather_electr...,204c21efe0b39603ad6e2ca640ef5896646ab1a9


## 14. Compact baseline table

In [15]:

compact = summary_df[
    [
        "Dataset",
        "Horizon",
        "OfficialStyle_MSE",
        "OfficialStyle_MAE",
        "FullStride1_MSE",
        "FullStride1_MAE",
        "BestEpoch",
        "BestValMSE",
    ]
].copy()

display(compact)

compact.to_csv(
    RESULT_ROOT
    / "compact_baselines.csv",
    index=False,
)


,Dataset,Horizon,OfficialStyle_MSE,OfficialStyle_MAE,FullStride1_MSE,FullStride1_MAE,BestEpoch,BestValMSE
0,Electricity,96,0.129935,0.222482,0.129977,0.222527,97,0.110311
1,Electricity,192,0.148336,0.241681,0.149116,0.242038,48,0.126097
2,Electricity,336,0.164611,0.258211,0.165690,0.258694,69,0.140645
3,Electricity,720,0.202358,0.292115,0.203216,0.292492,39,0.173104
4,Weather,96,0.150419,0.198269,0.149690,0.197816,65,0.390229
5,Weather,192,0.195386,0.241418,0.194775,0.240853,45,0.456669
6,Weather,336,0.247370,0.282072,0.246900,0.281548,19,0.547002
7,Weather,720,0.319053,0.334466,0.320894,0.334351,23,0.653166


## 15. Optional comparison with the old Experiment 13 screening baseline

In [16]:

OLD_SUMMARY_CANDIDATES = [
    Path(
        "/data/dataset/strong_forecaster/"
        "multidataset_crossfit_screening/"
        "summary.csv"
    ),
    Path(
        "/data/dataset/strong_forecaster/"
        "multidataset_crossfit_screening/"
        "screening_summary.csv"
    ),
]

old_path = next(
    (
        p
        for p in OLD_SUMMARY_CANDIDATES
        if p.is_file()
    ),
    None,
)

if old_path is None:
    print(
        "Old Experiment 13 summary not found. "
        "This comparison is optional."
    )
else:
    old = pd.read_csv(old_path)

    print(
        "Old screening summary:",
        old_path,
    )

    # Handle the column names used in our previous notebooks.
    direct_col = next(
        (
            c
            for c in [
                "PatchTST_MSE",
                "Direct_MSE",
                "DirectMSE",
            ]
            if c in old.columns
        ),
        None,
    )

    horizon_col = next(
        (
            c
            for c in [
                "Horizon",
                "PredLen",
                "pred_len",
            ]
            if c in old.columns
        ),
        None,
    )

    dataset_col = next(
        (
            c
            for c in [
                "Dataset",
                "dataset",
            ]
            if c in old.columns
        ),
        None,
    )

    if (
        direct_col is None
        or horizon_col is None
        or dataset_col is None
    ):
        print(
            "Could not identify the old direct-baseline columns. "
            "Available columns:"
        )
        print(
            old.columns.tolist()
        )
    else:
        old_sub = old[
            old[
                dataset_col
            ].isin(
                DATASETS
            )
        ][
            [
                dataset_col,
                horizon_col,
                direct_col,
            ]
        ].copy()

        old_sub.columns = [
            "Dataset",
            "Horizon",
            "OldScreening_MSE",
        ]

        comparison = compact.merge(
            old_sub,
            on=[
                "Dataset",
                "Horizon",
            ],
            how="left",
        )

        comparison[
            "StrongGainVsOld_pct"
        ] = (
            100.0
            * (
                comparison[
                    "OldScreening_MSE"
                ]
                - comparison[
                    "FullStride1_MSE"
                ]
            )
            / comparison[
                "OldScreening_MSE"
            ]
        )

        display(comparison)

        comparison.to_csv(
            RESULT_ROOT
            / "comparison_vs_exp13.csv",
            index=False,
        )


Old screening summary: /data/dataset/strong_forecaster/multidataset_crossfit_screening/summary.csv


,Dataset,Horizon,OfficialStyle_MSE,OfficialStyle_MAE,FullStride1_MSE,FullStride1_MAE,BestEpoch,BestValMSE,OldScreening_MSE,StrongGainVsOld_pct
0,Electricity,96,0.129935,0.222482,0.129977,0.222527,97,0.110311,0.196270,33.776385
1,Electricity,192,0.148336,0.241681,0.149116,0.242038,48,0.126097,0.198762,24.977493
2,Electricity,336,0.164611,0.258211,0.165690,0.258694,69,0.140645,0.220282,24.782742
3,Electricity,720,0.202358,0.292115,0.203216,0.292492,39,0.173104,0.271241,25.078996
4,Weather,96,0.150419,0.198269,0.149690,0.197816,65,0.390229,0.172091,13.016606
5,Weather,192,0.195386,0.241418,0.194775,0.240853,45,0.456669,0.217632,10.502482
6,Weather,336,0.247370,0.282072,0.246900,0.281548,19,0.547002,0.273997,9.889319
7,Weather,720,0.319053,0.334466,0.320894,0.334351,23,0.653166,0.351578,8.727437



# 16. Experiment 18 decision rule

이번 실험에서는 retrieval 결과를 만들지 않습니다.

확인할 것은 다음 두 가지입니다.

### 1. Official-style baseline이 충분히 강한가?

공식 script와 동일한 architecture, split, optimizer, schedule을 사용했기 때문에
재현값이 통상적인 PatchTST 결과와 비슷한 범위인지 확인합니다.

### 2. FullStride1 baseline을 최종 direct 기준으로 고정

Experiment 19에서는 각 조건의

$$
\texttt{FullStride1\_MSE}
$$

와 정확히 동일한 test window에 retrieval augmentation을 적용합니다.

즉 Experiment 19의 비교는

$$
\boxed{
\text{Official PatchTST FullStride1}
\quad \text{vs.} \quad
\text{Official PatchTST FullStride1 + Frozen Memory}
}
$$

가 됩니다.

---

## 중요한 연구 원칙

Experiment 18 결과를 보고 retrieval architecture나 gate를 변경하지 않습니다.

Weather와 Electricity에서 strong PatchTST가 확정되면,
Experiment 19는 Experiment 16/17과 동일한 frozen augmentation을 그대로 적용합니다.


## 17. Saved artifacts

In [17]:

print("Result root:", RESULT_ROOT)

for p in sorted(
    RESULT_ROOT.rglob("*")
):
    if p.is_file():
        print(
            p.relative_to(
                RESULT_ROOT
            )
        )


Result root: /data/dataset/strong_forecaster/weather_electricity_official_patchtst_baselines
artifacts/dataset_protocol.csv
artifacts/official_repo_commit.txt
compact_baselines.csv
comparison_vs_exp13.csv
full_direct/electricity_L336_H192_official_recipe_seed2021.pt
full_direct/electricity_L336_H336_official_recipe_seed2021.pt
full_direct/electricity_L336_H720_official_recipe_seed2021.pt
full_direct/electricity_L336_H96_official_recipe_seed2021.pt
full_direct/weather_L336_H192_official_recipe_seed2021.pt
full_direct/weather_L336_H336_official_recipe_seed2021.pt
full_direct/weather_L336_H720_official_recipe_seed2021.pt
full_direct/weather_L336_H96_official_recipe_seed2021.pt
history/electricity_L336_H192_official_recipe_seed2021.csv
history/electricity_L336_H336_official_recipe_seed2021.csv
history/electricity_L336_H720_official_recipe_seed2021.csv
history/electricity_L336_H96_official_recipe_seed2021.csv
history/weather_L336_H192_official_recipe_seed2021.csv
history/weather_L336_H336_o